# Introduction to NumPy

**NumPy** (Numerical Python) is the foundation of scientific computing in Python. It provides the `ndarray` — an n-dimensional array — which is dramatically faster than a Python list for numerical operations.

You've already been using NumPy indirectly: Pandas DataFrames are built on top of NumPy arrays. Understanding NumPy gives you insight into what's running under the hood of your DataFrames, and unlocks a whole new set of capabilities for numerical computing.

In this notebook we'll cover the fundamentals: creating arrays, understanding why they're fast, doing maths on them, and exploring their methods.

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain why NumPy arrays are faster than Python lists

2. Create NumPy arrays using `np.array()`, `np.zeros()`, `np.ones()`, `np.arange()`, and `np.random.rand()`

3. Perform element-wise arithmetic on arrays

4. Understand and apply **broadcasting** to operate on arrays of different shapes

5. Index and slice 1D and 2D NumPy arrays

6. Use aggregation methods: `.sum()`, `.mean()`, `.std()`, `.min()`, `.max()`, `.argmin()`, `.argmax()`

7. Use `np.where()` to find elements that meet a condition

8. Create a **pivot table** in Pandas using `pd.pivot_table()`

---

## Why NumPy Arrays Are Fast

From a high level, a NumPy array looks like a list — it stores a bunch of values in a container. But under the hood, there are two key differences that make NumPy dramatically faster:

1. **Contiguous memory**: All values are stored in one block of memory. Python lists scatter their elements around memory, causing [overhead](https://pythonspeed.com/articles/python-integers-memory/) when accessing them.

2. **Homogeneous type**: Every element in a NumPy array is the same data type (e.g. all `float64`). Python lists can hold mixed types, so Python has to check the type of every element before operating on it.

These properties enable **vectorisation** — the CPU can process multiple values in a single operation, rather than looping through them one by one.

Let's time it:

In [1]:
# Standard import — always use 'np' as the alias
import numpy as np

# --- Create a NumPy array and a Python list of the same values ---
numpy_array = np.arange(0, 1_000_000)
python_list = range(1_000_000)

In [2]:
# NumPy sum — uses vectorised operations
%timeit np.sum(numpy_array)

92.8 μs ± 144 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [3]:
# Python built-in sum on a list
%timeit sum(python_list)

7.88 ms ± 108 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [4]:
# Python built-in sum on a NumPy array — slower than np.sum()!
# NumPy arrays are optimised for NumPy operations, not Python built-ins.
%timeit sum(numpy_array)

32.4 ms ± 594 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


NumPy's `np.sum()` is roughly 10–50x faster than Python's `sum()`. The gap grows even larger for more complex operations. 

*Always use NumPy's own methods when working with NumPy arrays.*

---

## Creating NumPy Arrays

The most common way to create a NumPy array is `np.array()`, which accepts a list or tuple. NumPy infers the data type automatically, or you can specify it.

In [6]:
# Create from a list — dtype is inferred as int64
arr_from_list = np.array([1, 2, 3, 4, 5])

# Create from a tuple with an explicit dtype
arr_from_tuple = np.array((1, 2, 3, 4, 5), dtype=np.int32)

print(arr_from_list)
print(arr_from_tuple)

[1 2 3 4 5]
[1 2 3 4 5]


Use `.shape` and `.dtype` to inspect an array — just like `.shape` on a DataFrame.

In [7]:
# .shape returns (rows,) for 1D arrays, (rows, cols) for 2D
print(arr_from_list.shape)
print(arr_from_tuple.shape)

# .dtype tells you the type of the elements
print(arr_from_list.dtype)
print(arr_from_tuple.dtype)

(5,)
(5,)
int64
int32


NumPy enforces **homogeneity** — if you mix types, everything is cast to the most general type. Mixing strings and numbers turns everything into strings (Unicode).

In [8]:
# Mixing strings and numbers — everything becomes a string
mixed_arr = np.array(['1', 2, 3, '10', 5])
print(mixed_arr.dtype)   # 'U...' means Unicode string
print(type(mixed_arr[1]))  # The integer 2 is now a string

<U21
<class 'numpy.str_'>


In [9]:
# You can cast numeric strings to int32 — this works
castable_arr = np.array([1, 2, 3, '10', 5], dtype=np.int32)
print(castable_arr.dtype)

int32


In [ ]:
# This will raise a ValueError — 'bozo' cannot be cast to int32

# bad_arr = np.array([1, 2, 3, 'bozo', 5], dtype=np.int32)  # <-- uncomment to see error

### Other Useful Array Constructors

NumPy provides several functions for creating common arrays without having to list every value.

In [10]:
# Matrix of zeros — 3 rows, 4 columns
np.zeros((3, 4))

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

In [11]:
# Matrix of ones — 10 rows, 5 columns
np.ones((10, 5))

array([[1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.]])

In [12]:
# Identity matrix — 1s on the diagonal, 0s everywhere else
np.identity(5)

array([[1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1.]])

In [13]:
# 2x2 array of random floats between 0 and 1
np.random.rand(2, 2)

array([[0.1437354 , 0.94189968],
       [0.02410167, 0.7606311 ]])

In [14]:
# Range array: start=0, stop=20, step=0.5 — like Python's range() but with floats
np.arange(0, 20, 0.5)

array([ 0. ,  0.5,  1. ,  1.5,  2. ,  2.5,  3. ,  3.5,  4. ,  4.5,  5. ,
        5.5,  6. ,  6.5,  7. ,  7.5,  8. ,  8.5,  9. ,  9.5, 10. , 10.5,
       11. , 11.5, 12. , 12.5, 13. , 13.5, 14. , 14.5, 15. , 15.5, 16. ,
       16.5, 17. , 17.5, 18. , 18.5, 19. , 19.5])

---

## Array Arithmetic

All standard arithmetic operators (`+`, `-`, `*`, `/`, `**`, `%`) work on NumPy arrays and are applied **element-wise**. No loops needed.

In [15]:
# 1D element-wise addition
first_arr = np.array([1, 2, 3, 4])
second_arr = np.array([5, 6, 7, 8])
first_arr + second_arr

array([ 6,  8, 10, 12])

In [17]:
# 2D element-wise multiplication
first_2d = np.array([[1, 2], [3, 4]])
second_2d = np.array([[5, 6], [7, 8]])

print(first_2d)
print(second_2d)
first_2d * second_2d

[[1 2]
 [3 4]]
[[5 6]
 [7 8]]


array([[ 5, 12],
       [21, 32]])

### Broadcasting

**Broadcasting** is what happens when you apply an operation between a NumPy array and a single scalar value. NumPy "broadcasts" the scalar to match the shape of the array, then applies the operation element-wise.

Think of it as automatically expanding the scalar to fill the array.

In [18]:
arr = np.array([[1, 2], [3, 4]])

# --- Scalar broadcasting ---
print(arr - 4)   # Subtract 4 from every element
print(arr * 5)   # Multiply every element by 5
print(arr % 3)   # Remainder when every element is divided by 3

[[-3 -2]
 [-1  0]]
[[ 5 10]
 [15 20]]
[[1 2]
 [0 1]]


Broadcasting also works along rows or columns — giving you fine-grained control over which values to apply to which parts of the array.

In [19]:
arr = np.array([[1, 2], [3, 4]])

# Subtract 4 from column 0 and 5 from column 1
print(arr - [4, 5])

[[-3 -3]
 [-1 -1]]


In [20]:
# Subtract 4 from row 0 and 5 from row 1
print(arr - [[4], [5]])

[[-3 -2]
 [-2 -1]]


See the full [broadcasting rules](https://numpy.org/doc/stable/user/basics.broadcasting.html) for more detail.

---

## Indexing NumPy Arrays

NumPy arrays don't have `.loc[]` or `.iloc[]` — you index directly using **comma-separated row and column selectors** inside square brackets.

The format is: `array[row_selector, column_selector]`

Use `:` as a wildcard meaning "all".

In [21]:
# Create a 5x4 array using arange + reshape
range_arr = np.arange(0, 20, 1).reshape(5, 4)
range_arr

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [12, 13, 14, 15],
       [16, 17, 18, 19]])

In [22]:
# All rows, column at index 2
range_arr[:, 2]

array([ 2,  6, 10, 14, 18])

In [25]:
# First 2 rows, all columns (no column selector = all columns)
range_arr[0:2]

array([[0, 1, 2, 3],
       [4, 5, 6, 7]])

In [24]:
# Rows 0-1, columns 1-2
range_arr[0:2, 1:3]

array([[1, 2],
       [5, 6]])


---

## Aggregation Methods

NumPy arrays have the same statistical methods you've seen on Pandas columns: `.sum()`, `.mean()`, `.std()`, `.min()`, `.max()`. The key difference is the `axis` parameter, which controls the direction of the calculation.

- `axis=0` → operate **down the rows** (result per column)

- `axis=1` → operate **across columns** (result per row)

- No `axis` → operate on the **entire array**

In [26]:
# Sum down the rows — gives the total for each column
print('Column totals:', range_arr.sum(axis=0))

# Sum across columns — gives the total for each row
print('Row totals:   ', range_arr.sum(axis=1))

# Grand total of all elements
print('Grand total:  ', range_arr.sum())

Column totals: [40 45 50 55]
Row totals:    [ 6 22 38 54 70]
Grand total:   190


In [27]:
# Mean, std, max, min — all support axis parameter
print('Column means: ', range_arr.mean(axis=0))
print('Column std:   ', range_arr.std(axis=0))
print('Column max:   ', range_arr.max(axis=0))
print('Column min:   ', range_arr.min(axis=0))

Column means:  [ 8.  9. 10. 11.]
Column std:    [5.65685425 5.65685425 5.65685425 5.65685425]
Column max:    [16 17 18 19]
Column min:    [0 1 2 3]


### Finding the Index of Min/Max: `argmin()` and `argmax()`

If you want to know **where** the min or max occurs (not what the value is), use `.argmin()` or `.argmax()`.

In [28]:
# Index of the minimum value in each column (row index of the minimum)
print('Row index of column minimums:', range_arr.argmin(axis=0))

# Index of the maximum value in each column
print('Row index of column maximums:', range_arr.argmax(axis=0))

# Index of the overall minimum (flattened index)
print('Overall minimum at index:', range_arr.argmin())

# Index of the overall maximum
print('Overall maximum at index:', range_arr.argmax())

Row index of column minimums: [0 0 0 0]
Row index of column maximums: [4 4 4 4]
Overall minimum at index: 0
Overall maximum at index: 19


### Other Useful Methods

In [29]:
# Cumulative sum down the rows
range_arr.cumsum(axis=0)

array([[ 0,  1,  2,  3],
       [ 4,  6,  8, 10],
       [12, 15, 18, 21],
       [24, 28, 32, 36],
       [40, 45, 50, 55]])

In [30]:
# Flatten a 2D array into 1D — two equivalent methods
print(range_arr.flatten())
print(range_arr.ravel())  # ravel() returns a view if possible; flatten() always returns a copy

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


---

## Finding Elements with `np.where()`

`np.where(condition)` returns the **indices** of the elements that satisfy the condition — similar to boolean masking in Pandas, but giving you positions instead of filtered rows.

In [31]:
my_arr = np.array([2, 4, 6, 8, 24, 3, 8, 9, 12])

# Returns the indices where the condition is True
print('Indices where value <= 2:', np.where(my_arr <= 2))
print('Indices where value == 8:', np.where(my_arr == 8))
print('Indices where value > 6: ', np.where(my_arr > 6))

Indices where value <= 2: (array([0]),)
Indices where value == 8: (array([3, 6]),)
Indices where value > 6:  (array([3, 4, 6, 7, 8]),)


---

## NumPy and Pandas Work Together

Most NumPy array methods have direct equivalents on Pandas columns (Series). For example:

- `array.argmax()` on NumPy → `series.idxmax()` on Pandas

- `array.cumsum()` → `series.cumsum()`

- `np.where()` → Pandas boolean masking

Understanding NumPy methods gives you a deeper toolkit for working with both arrays and DataFrames.

## Pivot Tables in Pandas

A **pivot table** summarises a DataFrame by computing an aggregate statistic (default: mean) across the combinations of two categorical variables. If you've used Excel pivot tables, this is the same concept.

We use `pd.pivot_table()` — it builds on NumPy operations internally.

In [32]:
# Load the red wine dataset for the pivot table demo
import pandas as pd
red_wines_df = pd.read_csv('../data/winequality-red.csv', sep=';')
red_wines_df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


`pd.cut()` is a useful companion to pivot tables. It converts a **continuous column** into **categorical bins** — for example, grouping fixed acidity values into ranges like (4, 5], (5, 6], etc.

In [33]:
# Preview what pd.cut() produces — bins of size 1 across the fixed acidity range
pd.cut(red_wines_df['fixed acidity'], bins=np.arange(4, 17)).head()

0      (7, 8]
1      (7, 8]
2      (7, 8]
3    (11, 12]
4      (7, 8]
Name: fixed acidity, dtype: category
Categories (12, interval[int64, right]): [(4, 5] < (5, 6] < (6, 7] < (7, 8] ... (12, 13] < (13, 14] < (14, 15] < (15, 16]]

In [34]:
# Create labelled bins and add as a new column
bin_edges = np.arange(4, 17)
fa_bins = pd.cut(red_wines_df['fixed acidity'], bins=bin_edges, labels=bin_edges[:-1])
fa_bins.name = 'fa_bin'

# --- Pandas 3.0 note: use pd.concat() instead of DataFrame.append() ---
red_wines_df = pd.concat([red_wines_df, fa_bins], axis=1)
red_wines_df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,fa_bin
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,7
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,7
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,7
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,11
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,7


In [35]:
# Pivot table: mean residual sugar by quality (rows) and fixed acidity bin (columns)
pd.pivot_table(red_wines_df, values='residual sugar', index='quality', columns='fa_bin')

fa_bin,4,5,6,7,8,9,10,11,12,13,14,15
quality,,,,,,,,,,,,
3,NaN,NaN,1,3,3,NaN,1,2,NaN,NaN,NaN,NaN
4,1,5,2,2,2,2,2,1,4,NaN,NaN,NaN
5,1,1,2,2,2,2,3,2,2,3,NaN,5
6,2,2,2,2,2,2,2,2,2,2,1,NaN
7,2,1,2,2,2,2,2,2,4,2,2,3
8,2,1,NaN,2,1,2,3,5,2,NaN,NaN,NaN


In [36]:
# Specify a different aggregation function — here the maximum instead of mean
pd.pivot_table(red_wines_df, values='residual sugar', index='quality',
               columns='fa_bin', aggfunc=np.max)

fa_bin,4,5,6,7,8,9,10,11,12,13,14,15
quality,,,,,,,,,,,,
3,NaN,NaN,1,5,3,NaN,2,2,NaN,NaN,NaN,NaN
4,2,12,5,4,6,3,3,1,4,NaN,NaN,NaN
5,1,2,7,8,7,13,15,5,4,4,NaN,7
6,4,13,10,5,5,11,15,6,4,3,1,NaN
7,2,2,6,8,6,8,6,4,5,2,2,3
8,2,1,NaN,3,1,2,6,5,2,NaN,NaN,NaN


---

## Key Takeaways

- **NumPy arrays** are faster than Python lists because of contiguous memory and homogeneous types — these enable **vectorisation**

- Always use NumPy's own methods (`np.sum()`, etc.) on NumPy arrays, not Python built-ins

- Arrays support element-wise arithmetic: `+`, `-`, `*`, `/`, `**`, `%`

- **Broadcasting** lets you apply a scalar or smaller array to a larger one without loops

- Index arrays with `[row, col]` notation — use `:` to mean "all"

- `axis=0` means "down the rows" (per column); `axis=1` means "across columns" (per row)

- `np.where()` returns the **indices** of elements matching a condition

- Pandas DataFrames are built on NumPy — most NumPy methods have Pandas equivalents

- Use `pd.cut()` to bin continuous data, and `pd.pivot_table()` to summarise across two categorical variables

---

## Check Your Understanding

1. What are the two main reasons NumPy arrays are faster than Python lists?

2. You create `arr = np.array([1, 2, 'three', 4])`. What is `arr.dtype`? Why?

3. You have a 3×4 NumPy array. What does `arr.sum(axis=1)` return — a single number, 3 values, or 4 values?

4. What is the difference between `arr.argmax()` and `arr.max()`?

5. You want to find all positions in an array where the value is greater than 100. Which NumPy function would you use?

### **Ready to practice?** Open [Notebook_2](02_Numpy_Practice.ipynb)

In [39]:
arrr = np.array([[1, 2, 3, 12], [4, 5, 6, 11], [7, 8, 9, 10]])

In [43]:
arrr.sum(axis=1)

array([18, 26, 34])